# Creating our basic Research Agent
Companion notebook for the **Creating our basic Research Agent** section of Chapter 4 (this is the notebook version of `state.py` / `nodes.py` / `graph.py` / `main.py`).

This is version 1 of the research agent: a simple, sequential graph -- `prepare_query -> search_sources -> summarize_results`.

> Add your key(s) to a `.env` file next to this notebook (see `.env.example`) to get real LLM-generated summaries. Without a key the notebook still runs end-to-end -- it just skips the LLM call and says so.

In [1]:
from pydantic import BaseModel, Field


class ResearchState(BaseModel):
    """State definition for the research agent workflow."""
    query: str = Field(default="", description="The user's research question or search query")
    candidate_urls: list[str] = Field(default_factory=list, description="Potentially relevant source URLs")
    page_texts: list[str] = Field(default_factory=list, description="Fetched page content from the candidate URLs")
    draft_summary: str = Field(default="", description="Preliminary summary from the gathered page texts")
    final_report: str = Field(default="", description="The finalized research report")
    errors: list[str] = Field(default_factory=list, description="Error messages from the workflow")

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.retrievers import WikipediaRetriever
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENAI_API_KEY from a local .env file, if present

wiki = WikipediaRetriever()
# Falls back to a dummy key so the client can be constructed even when OPENAI_API_KEY
# is not set; the guard inside summarize_results() stops it from ever being called.
llm = ChatOpenAI(model_name="gpt-4", temperature=0.0, api_key=os.getenv("OPENAI_API_KEY", "sk-not-set"))

C:\Users\himan\AppData\Local\Temp\ipykernel_49532\2419585624.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


**Node 1 - `prepare_query`**: removes whitespace from the query. Nodes return a *partial* state update (a dict of just the keys they changed), which is the pattern LangGraph expects.

In [3]:
def prepare_query(state: ResearchState) -> dict:
    cleaned = state.query.strip()
    return {"query": cleaned}

**Node 2 - `search_sources`**: searches Wikipedia for the query and stores the URLs and page text.

In [4]:
def search_sources(state: ResearchState) -> dict:
    query = state.query
    if not query:
        return {"candidate_urls": [], "page_texts": []}

    try:
        docs = wiki.invoke(query)
    except Exception as e:
        return {"candidate_urls": [], "page_texts": [], "errors": [f"Wikipedia error: {e}"]}

    urls = [doc.metadata.get("url", doc.metadata.get("title", "")) for doc in docs]
    texts = [doc.page_content for doc in docs]
    return {"candidate_urls": urls, "page_texts": texts}

**Node 3 - `summarize_results`**: asks the LLM to summarize the gathered page texts.

In [5]:
def summarize_results(state: ResearchState) -> dict:
    combined = "\n\n".join(state.page_texts)

    if not combined:
        return {"final_report": "No content found."}

    if not os.getenv("OPENAI_API_KEY"):
        return {"final_report": "[Skipped LLM call: set OPENAI_API_KEY in your .env to generate a real summary]"}

    prompt = f"Summarize the key points and cite sources if needed:\n\n{combined}"
    try:
        summary = llm.invoke(prompt)
        return {"final_report": summary.content}
    except Exception as e:
        return {"final_report": f"[LLM call failed: {e}]"}

Now register the nodes and wire the (static) edges: `prepare_query -> search_sources -> summarize_results`.

In [6]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ResearchState)

builder.add_node("prepare_query", prepare_query)
builder.add_node("search_sources", search_sources)
builder.add_node("summarize_results", summarize_results)

builder.add_edge(START, "prepare_query")
builder.add_edge("prepare_query", "search_sources")
builder.add_edge("search_sources", "summarize_results")
builder.add_edge("summarize_results", END)

graph = builder.compile()

In [7]:
initial_state = {
    "query": "Latest research on quantum computing",
    "candidate_urls": [],
    "page_texts": [],
    "draft_summary": "",
    "final_report": "",
    "errors": [],
}

final_state = graph.invoke(initial_state)

print(f"Query: {final_state['query']}")
print(f"Sources found: {len(final_state['candidate_urls'])}")
for url in final_state["candidate_urls"]:
    print(f"  - {url}")
print(f"\nFinal report:\n{final_state['final_report']}")

Query: Latest research on quantum computing
Sources found: 3
  - Google Quantum AI
  - Timeline of quantum computing and communication
  - Xanadu Quantum Technologies

Final report:
[Skipped LLM call: set OPENAI_API_KEY in your .env to generate a real summary]
